# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UmairMehfooz/Ml-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1 — Refreshing Pages Actually Works

### Methodology question

The paper reports evidence that refreshing pages is associated with improved performance. My question is how the outcome and comparison group were defined.

Specifically, I would want to know:

1. How was a "refresh" identified in the data?
2. What time window was used to measure performance before and after the refresh?
3. Were refreshed pages compared with similar pages that were not refreshed during the same period?

This matters because an observed improvement after a refresh does not by itself establish that the refresh caused the improvement. Search demand, seasonality, ranking changes, or other simultaneous changes could also affect the outcome.

I would therefore treat the finding as evidence of an observed relationship unless the study design supports a stronger causal claim.

## Finding 2 — What Will Improve Next Month?

### Methodology question

The paper includes a machine-learning analysis intended to identify content that may improve in the following period.

My main methodology question is whether the validation design prevents information from the same client or content portfolio appearing in both training and evaluation data.

If pages from the same client can occur in both sets, a model may learn client-specific patterns rather than generalizable relationships. I would therefore want to know whether validation was grouped by client or otherwise designed to prevent this form of overlap.

This is important because the strength of a model's validation result depends not only on the metric but also on whether the evaluation setup resembles the decision it is intended to support.

I would treat the reported result as measured evidence under the stated validation design, rather than assuming it generalizes beyond that design.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

rel = "hf://datasets/FlyRank/internship-warehouse"

march_path = (
    f"{rel}/fact_content_daily_performance/"
    "month=2026-03/**/*.parquet"
)

april_path = (
    f"{rel}/fact_content_daily_performance/"
    "month=2026-04/**/*.parquet"
)

print("Setup complete.")

Setup complete.


In [3]:
march_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(sessions_organic) AS sessions_organic,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
FROM read_parquet('{march_path}')
WHERE gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
"""

march_features = con.sql(march_query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [4]:
april_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS april_impressions
FROM read_parquet('{april_path}')
WHERE gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
"""

april_outcome = con.sql(april_query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
model_df = march_features.merge(
    april_outcome,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

In [6]:
model_df["impression_change_pct"] = (
    (
        model_df["april_impressions"]
        - model_df["gsc_impressions"]
    )
    / model_df["gsc_impressions"]
) * 100

model_df["decline_label"] = (
    model_df["impression_change_pct"] < -20
).astype(int)

In [7]:
feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "sessions_organic",
    "ga4_engaged_sessions"
]

model_df = model_df.dropna(
    subset=feature_columns + ["decline_label"]
).reset_index(drop=True)

In [8]:
X = model_df[feature_columns]
y = model_df["decline_label"]

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

random_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

random_model.fit(
    X_train_random,
    y_train_random
)

random_probability = random_model.predict_proba(
    X_test_random
)[:, 1]

In [9]:
def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    order = np.argsort(scores)[::-1][:k]

    return y_true[order].mean()

random_precision_50 = precision_at_k(
    y_test_random,
    random_probability,
    k=50
)

print(
    "Random split Precision@50:",
    round(random_precision_50, 4)
)

Random split Precision@50: 0.68


In [10]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_df,
        model_df["decline_label"],
        groups=model_df["client_hash_id"]
    )
)

train_grouped = model_df.iloc[train_idx]
test_grouped = model_df.iloc[test_idx]

In [11]:
grouped_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

grouped_model.fit(
    train_grouped[feature_columns],
    train_grouped["decline_label"]
)

grouped_probability = grouped_model.predict_proba(
    test_grouped[feature_columns]
)[:, 1]

In [12]:
grouped_precision_50 = precision_at_k(
    test_grouped["decline_label"],
    grouped_probability,
    k=50
)

print(
    "Grouped split Precision@50:",
    round(grouped_precision_50, 4)
)

Grouped split Precision@50: 0.66


In [13]:
shared_clients = (
    set(train_grouped["client_hash_id"])
    &
    set(test_grouped["client_hash_id"])
)

print(
    "Shared clients:",
    len(shared_clients)
)

Shared clients: 0


In [14]:
validation_comparison = pd.DataFrame({
    "validation_design": [
        "Random row split",
        "Grouped by client"
    ],
    "precision_at_50": [
        random_precision_50,
        grouped_precision_50
    ]
})

validation_comparison

,validation_design,precision_at_50
0,Random row split,0.68
1,Grouped by client,0.66


## 2. My model under an honest split

I first evaluated Logistic Regression using a random row split as a weaker reference design. I then repeated the experiment using a grouped split by `client_hash_id`, ensuring that no client appeared in both training and test sets.

The random split produced a Precision@50 of **[RANDOM RESULT]**.

The client-grouped split produced a Precision@50 of **[GROUPED RESULT]**.

The grouped result is the more credible estimate for my intended use because the model should support decisions across client portfolios rather than rely on seeing pages from the same client during training.

The difference between the two results shows why validation design matters. A higher score under a random split would not necessarily mean the model is better; it could partly reflect the easier evaluation setup.

The grouped split had **0 shared clients** between train and test.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
leakage_audit = pd.DataFrame({
    "field": [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "sessions_organic",
        "ga4_engaged_sessions",
        "april_impressions",
        "decline_label"
    ],
    "used_as_feature": [
        True,
        True,
        True,
        True,
        True,
        False,
        False
    ],
    "timing": [
        "March decision window",
        "March decision window",
        "March decision window",
        "March decision window",
        "March decision window",
        "April outcome window",
        "Derived from April outcome"
    ],
    "leakage_risk": [
        "Low",
        "Low",
        "Low",
        "Low",
        "Low",
        "High if used as feature",
        "High if used as feature"
    ]
})

leakage_audit

,field,used_as_feature,timing,leakage_risk
0,gsc_impressions,True,March decision window,Low
1,gsc_clicks,True,March decision window,Low
2,gsc_avg_position,True,March decision window,Low
3,sessions_organic,True,March decision window,Low
4,ga4_engaged_sessions,True,March decision window,Low
5,april_impressions,False,April outcome window,High if used as feature
6,decline_label,False,Derived from April outcome,High if used as feature


## 3. Leakage audit

The target is a future outcome: whether impressions decline by more than 20% in April.

All five model features are calculated from March data, which represents information available at the decision moment.

The April impression value is used only to construct the outcome label and is not included as a model feature.

I also exclude `decline_label`, `impression_change_pct`, and other future-window fields from the feature matrix.

This means the model does not receive the outcome it is being asked to predict.

The main remaining limitation is that the validation uses one March-to-April transition. A stronger future analysis would repeat the experiment across multiple historical time windows.

In [16]:
future_or_label_columns = [
    "april_impressions",
    "impression_change_pct",
    "decline_label"
]

leaked_features = set(feature_columns) & set(
    future_or_label_columns
)

print(
    "Future/label columns accidentally used as features:",
    leaked_features
)

Future/label columns accidentally used as features: set()


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

### Claim 1

**Too strong:** "The model predicts which pages will decline."

**Evidence-safe:** "The model ranks pages by estimated decline risk."

### Claim 2

**Too strong:** "The model is 66% accurate."

**Evidence-safe:** "The model achieved a Precision@50 of 0.660 on the held-out test split."

### Claim 3

**Too strong:** "The model tells us which pages need refreshing."

**Evidence-safe:** "The model provides decision support for prioritizing pages for human review."

### Claim 4

**Too strong:** "Clicks prevent content decline."

**Evidence-safe:** "Higher Google Search clicks were associated with lower predicted decline risk in this model."

### Claim 5

**Too strong:** "The model will generalize to all clients."

**Evidence-safe:** "The grouped validation evaluates whether the model can rank pages when clients are held out from training, but further time-window validation would be needed to establish broader robustness."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.